In [16]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
import numpy as np

### Load Data

In this version, we filtered the data to eliminate things the model struggled with:
1. Code heavy answers
2. Long complex prompts

In [17]:
df = pd.read_csv("prompt_answer_pairs_clean.csv")
print(f"Original: {len(df)} rows")

# 1. Filter out rows where answer is mostly code blocks
df = df[~df["answer"].str.contains(r'(\[CODE_BLOCK_\d+\].*){3,}', regex=True)]
print(f"After code block filter: {len(df)} rows")

# 2. Filter out very long prompts (over 50 words)
df = df[df["prompt"].str.split().str.len() <= 50]
print(f"After long prompt filter: {len(df)} rows")

# 3. Filter out very short prompts (less than 3 words)
df = df[df["prompt"].str.split().str.len() >= 3]
print(f"After short prompt filter: {len(df)} rows")

# 4. Filter out very long answers (over 400 words)
df = df[df["answer"].str.split().str.len() <= 400]
print(f"After long answer filter: {len(df)} rows")

# 5. Drop any rows where answer still has only code blocks and nothing else
df = df[~df["answer"].str.fullmatch(r'(\[CODE_BLOCK_\d+\]\s*)+', na=False)]
print(f"After code-only answer filter: {len(df)} rows")

df.to_csv("prompt_answer_pairs_filtered.csv", index=False, encoding="utf-8")

Original: 8058 rows


/tmp/ipykernel_6480/387435842.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["answer"].str.contains(r'(\[CODE_BLOCK_\d+\].*){3,}', regex=True)]


After code block filter: 7510 rows
After long prompt filter: 4389 rows
After short prompt filter: 4357 rows
After long answer filter: 4056 rows
After code-only answer filter: 3993 rows


In this version we build context, building history from the prior turns

In [18]:
def build_context_input(group):
    rows = group.reset_index(drop=True)
    context_inputs = []

    for i, row in rows.iterrows():
        # Only use last 2 prior turns
        history = ""
        start = max(0, i - 2)
        # Build history by appending previous
        for j in range(start, i):
            prev = rows.iloc[j]
            history += f"User: {prev['prompt']} Assistant: {prev['answer']} "

        # If there is a history prepend it, if not just use answer
        if history:
            context_input = f"predict prompt: {history.strip()} Current answer: {row['answer']}"
        else:
            context_input = f"predict prompt: {row['answer']}"

        context_inputs.append(context_input)

    group = group.copy()
    group["context_input"] = context_inputs
    return group

# Splits df into groups by conversation url and then applys the context function independently
df = df.sort_values(["chatgpt_url", "conv_index"]).reset_index(drop=True)
df = df.groupby("chatgpt_url", group_keys=False).apply(build_context_input)

df.to_csv("prompt_answer_pairs_context.csv", index=False, encoding="utf-8")

/tmp/ipykernel_6480/1790556991.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("chatgpt_url", group_keys=False).apply(build_context_input)


In [19]:
df = pd.read_csv("prompt_answer_pairs_context.csv")[["context_input", "prompt", "answer"]].dropna()

# Convert to strings
df["prompt"] = df["prompt"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df["context_input"] = df["context_input"].astype(str).str.strip()

# 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

### Tokenize data

In [20]:
def tokenize_data(df):
  # Input is the answer, tokenizes answer text
  inputs = tokenizer(
      df["context_input"].tolist(),
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
    )
  # Target is the prompt, tokenizes prompt text
  targets = tokenizer(
      df["prompt"].tolist(),
      max_length=128,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  )

  labels = targets["input_ids"]
  # Replace padding tokens with -100 so model does not try to predict them
  labels[labels == tokenizer.pad_token_id] = -100

  # Create tokenized dataset
  dataset = torch.utils.data.TensorDataset(
      inputs["input_ids"],
      inputs["attention_mask"],
      labels
  )
  return dataset

In [21]:
# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")
# Model is a T5 language model
# T5 is a seq2seq model that is pretrained using a  using a denoising objective
model     = T5ForConditionalGeneration.from_pretrained("t5-small").to(torch.device("cuda"))

# Tokenize all datasets
train_dataset = tokenize_data(train_df)
val_dataset   = tokenize_data(val_df)
test_dataset  = tokenize_data(test_df)

# DataLoader does batching, shuffling and makes dataset iterable
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Evaluate how model is doing on validation set

In [22]:
def evaluate(loader):
  # Switch model to evaluation mode
  model.eval()
  total_loss = 0
  # Do not track gradients to save memory
  with torch.no_grad():
    # For every batch accumulate the loss
    for batch in loader:
        input_ids, attention_mask, labels = batch
        input_ids      = input_ids.to(torch.device("cuda"))
        attention_mask = attention_mask.to(torch.device("cuda"))
        labels         = labels.to(torch.device("cuda"))
        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        total_loss += loss.item()
  # Return the average loss per epoch
  return total_loss / len(loader)

### Train Model

In [23]:
# Load optimizer
optimizer   = torch.optim.AdamW(model.parameters(), lr=5e-5)
best_val    = float("inf")

for epoch in range(5):
  # Switch model to train mode
  model.train()
  total_train_loss = 0

  for i, batch in enumerate(train_loader):
    input_ids, attention_mask, labels = batch
    input_ids      = input_ids.to(torch.device("cuda"))
    attention_mask = attention_mask.to(torch.device("cuda"))
    labels         = labels.to(torch.device("cuda"))

    # Clear gradient from previous batch
    optimizer.zero_grad()
    # Calculates loss using T5 model
    loss = model(input_ids=input_ids,
                  attention_mask=attention_mask,
                  labels=labels).loss
    # Backpropagation
    loss.backward()
    # Gradient clipping (caps gradients at 1)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    # Updates model weights using gradients calculated by loss.backward
    optimizer.step()

    total_train_loss += loss.item()

    if (i + 1) % 50 == 0:
      print(f"  Epoch {epoch+1} | Step {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

  avg_train = total_train_loss / len(train_loader)
  avg_val   = evaluate(val_loader)
  print(f"\nEpoch {epoch+1}/5 | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

  # Gets best model by updating model saved when average validation loss is less
  if avg_val < best_val:
    best_val = avg_val
    model.save_pretrained("prompt_predictor")
    tokenizer.save_pretrained("prompt_predictor")

  Epoch 1 | Step 50/350 | Loss: 3.2859
  Epoch 1 | Step 100/350 | Loss: 3.7513
  Epoch 1 | Step 150/350 | Loss: 3.5556
  Epoch 1 | Step 200/350 | Loss: 3.8465
  Epoch 1 | Step 250/350 | Loss: 2.7203
  Epoch 1 | Step 300/350 | Loss: 3.6354
  Epoch 1 | Step 350/350 | Loss: 3.8366

Epoch 1/5 | Train Loss: 3.5278 | Val Loss: 3.2502


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 2 | Step 50/350 | Loss: 3.3335
  Epoch 2 | Step 100/350 | Loss: 3.2630
  Epoch 2 | Step 150/350 | Loss: 3.5814
  Epoch 2 | Step 200/350 | Loss: 3.1294
  Epoch 2 | Step 250/350 | Loss: 2.8999
  Epoch 2 | Step 300/350 | Loss: 2.8323
  Epoch 2 | Step 350/350 | Loss: 3.1091

Epoch 2/5 | Train Loss: 3.3015 | Val Loss: 3.1909


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 3 | Step 50/350 | Loss: 3.4490
  Epoch 3 | Step 100/350 | Loss: 2.8867
  Epoch 3 | Step 150/350 | Loss: 3.2519
  Epoch 3 | Step 200/350 | Loss: 3.3414
  Epoch 3 | Step 250/350 | Loss: 3.3817
  Epoch 3 | Step 300/350 | Loss: 3.4964
  Epoch 3 | Step 350/350 | Loss: 5.0250

Epoch 3/5 | Train Loss: 3.2202 | Val Loss: 3.1492


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 4 | Step 50/350 | Loss: 2.3926
  Epoch 4 | Step 100/350 | Loss: 3.1907
  Epoch 4 | Step 150/350 | Loss: 3.9643
  Epoch 4 | Step 200/350 | Loss: 2.9108
  Epoch 4 | Step 250/350 | Loss: 3.5958
  Epoch 4 | Step 300/350 | Loss: 3.4527
  Epoch 4 | Step 350/350 | Loss: 3.4439

Epoch 4/5 | Train Loss: 3.1284 | Val Loss: 3.1271


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 5 | Step 50/350 | Loss: 2.9880
  Epoch 5 | Step 100/350 | Loss: 4.0717
  Epoch 5 | Step 150/350 | Loss: 2.2306
  Epoch 5 | Step 200/350 | Loss: 3.2860
  Epoch 5 | Step 250/350 | Loss: 3.3604
  Epoch 5 | Step 300/350 | Loss: 3.3952
  Epoch 5 | Step 350/350 | Loss: 3.8517

Epoch 5/5 | Train Loss: 3.0792 | Val Loss: 3.1201


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Evaulate Model Performace with ROUGE
ROUGE is a metric that measures how simular predicted prompt is to real prompt by comparing n-gram overlap

- ROUGE-1: Counts overlap of individual words
- ROUGE-2: Counts overlap of word pairs
- ROUGE-L: Finds longest common sequence of words in order

In [24]:
model = T5ForConditionalGeneration.from_pretrained("prompt_predictor").to(torch.device("cuda"))
model.eval()

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
r1, r2, rL = [], [], []

for _, row in test_df.iterrows():
    # Tokenize input
    input_text = row["context_input"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(torch.device("cuda"))

    # Generate prediction
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    # Decode prediction back to text
    predicted = tokenizer.decode(output[0], skip_special_tokens=True)

    # Score predicted vs real prompt
    scores = scorer.score(row["prompt"], predicted)
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

# Print results
print(f"ROUGE-1:  {np.mean(r1):.4f}")
print(f"ROUGE-2:  {np.mean(r2):.4f}")
print(f"ROUGE-L:  {np.mean(rL):.4f}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

ROUGE-1:  0.2790
ROUGE-2:  0.1271
ROUGE-L:  0.2395


### Sample Predictions

In [25]:
for _, row in test_df.head(20).iterrows():
    input_text = row["context_input"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(torch.device("cuda"))

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    predicted = tokenizer.decode(output[0], skip_special_tokens=True)

    print(f"\nAnswer (truncated): {row['answer'][:200]}")
    print(f"Real prompt:        {row['prompt']}")
    print(f"Predicted prompt:   {predicted}")
    print(f"ROUGE-L:            {scorer.score(row['prompt'], predicted)['rougeL'].fmeasure:.4f}")


Answer (truncated): Here are the moods I would assign to the next 10 songs from the Billboard Year-End Hot 100 singles of 2017:"Say You Won't Let Go" - James Arthur: Happy, Calm"I'm the One" - DJ Khaled featuring Justin 
Real prompt:        Do the next 10 songs.
Predicted prompt:   What moods would you assign to the next 10 songs from the Billboard Year-End Hot 100 singles of 2017
ROUGE-L:            0.3200

Answer (truncated): While Visual Studio Code (VSCode) itself doesn't have an integrated UUID generator, you can easily create a UUID by using a VSCode extension or using an integrated terminal with a command-line utility
Real prompt:        How to generate a UUID in VSCode?
Predicted prompt:   Can you create a UUID using a VSCode extension?
ROUGE-L:            0.3750

Answer (truncated): Certainly, you can conditionally render an empty state message when isAvailable is false. You'd use Salesforce LWC's conditional rendering with template if:true or template if:false.Here's how you